# Car prices — data exploration
The dealership contains different brands, models and model years. I inspect price distributions, mileage, engine volume, year, body and registration before choosing transformations.

The original notebook removed price and mileage tails and older cars. Here I inspect those tails on training rows and retain them for an honest comparison. A normal-looking distribution alone does not establish regression assumptions.

**My notes:** update the observations below from the current training split.


## Setup and training-only split


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display, HTML

# Works when Jupyter starts in this folder, notebooks/, or the repository root.
locations = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (candidate for location in locations
     for candidate in (location, location / "regression" / "car_price_prediction")
     if (candidate / "car_prices_dataset" / "train" / "car_price.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository or car_price_prediction folder.")
DATA_DIR = PROJECT_ROOT / "car_prices_dataset"
TRAIN_PATH = DATA_DIR / "train" / "car_price.csv"
GROUND_TRUTH_DIR = DATA_DIR / "ground_truth"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
REPORTS_DIR = PROJECT_ROOT / "reports"
TARGET = "Price"
ID_COLUMN = "RowId"
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_columns", 20)


In [2]:
# Match these split/feature settings to training if you change them.
config = {"random_state": 42, "ground_truth_fraction": 0.2,
          "validation_fraction": 0.2, "reference_year": 2016}
display(pd.Series(config, name="diagnostic_configuration").to_frame())


,diagnostic_configuration
random_state,42.0
ground_truth_fraction,0.2
validation_fraction,0.2
reference_year,2016.0


In [3]:
from sklearn.model_selection import train_test_split

def prepare_data(config):
    """Assign stable source-row IDs, validate labels, deduplicate, then split."""
    raw = pd.read_csv(TRAIN_PATH)
    required = {"Brand", "Price", "Body", "Mileage", "EngineV", "Engine Type", "Registration", "Year", "Model"}
    missing = required - set(raw.columns)
    if missing:
        raise ValueError(f"Dataset is missing columns: {sorted(missing)}")
    if ID_COLUMN in raw:
        raise ValueError(f"Source column {ID_COLUMN} is reserved for generated row IDs")
    source_columns = raw.columns.tolist()
    raw.insert(0, ID_COLUMN, np.arange(1, len(raw) + 1))
    raw[TARGET] = pd.to_numeric(raw[TARGET], errors="coerce")
    valid_label = np.isfinite(raw[TARGET]) & (raw[TARGET] > 0)
    labeled = raw.loc[valid_label].copy()
    # Ignore generated IDs when identifying identical source records.
    duplicate = labeled.duplicated(subset=source_columns, keep="first")
    usable = labeled.loc[~duplicate].copy()
    if len(usable) < 10 or usable[TARGET].nunique() < 2:
        raise ValueError("Need at least ten labeled unique rows and two distinct positive prices")
    for key in ("ground_truth_fraction", "validation_fraction"):
        if not 0 < config[key] < 0.5:
            raise ValueError(f"{key} must be between 0 and 0.5")
    development, ground = train_test_split(usable, test_size=config["ground_truth_fraction"], random_state=config["random_state"])
    training, validation = train_test_split(development, test_size=config["validation_fraction"], random_state=config["random_state"])
    if config.get("cv_folds", 2) > len(training):
        raise ValueError("cv_folds exceeds training rows")
    assert not (set(training[ID_COLUMN]) & set(validation[ID_COLUMN])
                or set(training[ID_COLUMN]) & set(ground[ID_COLUMN])
                or set(validation[ID_COLUMN]) & set(ground[ID_COLUMN]))
    audit = {"source_rows": len(raw), "excluded_invalid_or_missing_prices": int((~valid_label).sum()),
             "excluded_exact_duplicates": int(duplicate.sum()), "usable_rows": len(usable),
             "training_rows": len(training), "validation_rows": len(validation), "ground_truth_rows": len(ground)}
    return training, validation, ground, audit


In [4]:
raw = pd.read_csv(TRAIN_PATH)
training, validation, ground, audit = prepare_data(config)
eda = training.copy()
display(pd.Series(audit, name="data_preparation").to_frame())
display(raw.head())
display(eda.describe(include="all").T)


,data_preparation
source_rows,4345
excluded_invalid_or_missing_prices,172
excluded_exact_duplicates,50
usable_rows,4123
training_rows,2638
validation_rows,660
ground_truth_rows,825


,Brand,Price,Body,Mileage,EngineV,Engine Type,Registration,Year,Model
0,BMW,4200.0,sedan,277,2.0,Petrol,yes,1991,320
1,Mercedes-Benz,7900.0,van,427,2.9,Diesel,yes,1999,Sprinter 212
2,Mercedes-Benz,13300.0,sedan,358,5.0,Gas,yes,2003,S 500
3,Audi,23000.0,crossover,240,4.2,Petrol,yes,2007,Q7
4,Toyota,18300.0,crossover,120,2.0,Petrol,yes,2011,Rav 4


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
RowId,2638.0,NaN,NaN,NaN,2172.612964,1244.35436,1.0,1084.25,2174.5,3231.25,4342.0
Brand,2638,7,Volkswagen,588,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Price,2638.0,NaN,NaN,NaN,19502.691873,25624.987894,600.0,7000.0,11700.0,21775.0,300000.0
Body,2638,6,sedan,1005,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Mileage,2638.0,NaN,NaN,NaN,163.688779,105.246194,0.0,90.0,156.0,230.0,940.0
EngineV,2542.0,NaN,NaN,NaN,2.852565,5.628672,0.6,1.8,2.2,3.0,99.99
Engine Type,2638,4,Diesel,1182,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Registration,2638,2,yes,2400,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Year,2638.0,NaN,NaN,NaN,2006.452995,6.562718,1969.0,2003.0,2007.0,2011.0,2016.0
Model,2638,272,E-Class,120,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Price distribution and target transformation
Prices have a long upper tail. The original log-price approach is retained as log1p/expm1 during model fitting. Comparison metrics are computed after predictions return to price units.

**My notes:** describe skewness and the effect of expensive cars on RMSE.


In [5]:
display(pd.DataFrame({"Price": eda.Price, "log1p_price": np.log1p(eda.Price)}).agg(["mean", "median", "std", "skew"]).T)
px.histogram(eda, x="Price", nbins=50, title="Training prices").show()
px.histogram(eda.assign(log1p_price=np.log1p(eda.Price)), x="log1p_price", nbins=50, title="Log-transformed prices").show()


,mean,median,std,skew
Price,19502.691873,11700.00000,25624.987894,4.561967
log1p_price,9.417285,9.36743,0.923688,0.224059


## Mileage, engine volume and model year
I inspect both raw-price and log-price relationships. EngineV values above 6.5 are suspicious under the original notebook's domain rule. Mileage tails and vintage cars remain valid observations unless a concrete data error is established.

**My notes:** record nonlinear patterns and suspicious engine measurements.


In [6]:
for column in ["Mileage", "EngineV", "Year"]:
    px.histogram(eda, x=column, nbins=40, title=f"Training {column}").show()
    px.scatter(eda.assign(log1p_price=np.log1p(eda.Price)), x=column, y="log1p_price", color="Brand", hover_data=[ID_COLUMN], title=f"{column} vs log price").show()
display(eda[["Price", "Mileage", "EngineV", "Year"]].quantile([0, .01, .5, .99, 1]))


,Price,Mileage,EngineV,Year
0.00,600.00,0.00,0.60,1969.0
0.01,1500.00,0.00,1.20,1987.0
0.50,11700.00,156.00,2.20,2007.0
0.99,129711.51,449.26,6.30,2016.0
1.00,300000.00,940.00,99.99,2016.0


## Categories, missingness and model cardinality
Brand, body, fuel/engine type and registration describe different market segments. Model has many categories; the baseline retains the original decision to omit it. Model-aware variants should be compared within training CV before adopting them.


In [7]:
display(eda.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
display(eda.select_dtypes(exclude="number").nunique().to_frame("unique_categories"))
for column in ["Brand", "Body", "Engine Type", "Registration"]:
    display(eda.groupby(column).Price.agg(["count", "median", "mean"]).sort_values("median"))
    px.box(eda, x=column, y="Price", title=f"Training prices by {column}").show()


,missing_count
EngineV,96
RowId,0
Brand,0
Price,0
Body,0
Mileage,0
Engine Type,0
Registration,0
Year,0
Model,0


,unique_categories
Brand,7
Body,6
Engine Type,4
Registration,2
Model,272


,count,median,mean
Brand,,,
Renault,284,7500.00,7912.754507
Mitsubishi,202,9550.00,11170.277723
Volkswagen,588,10500.00,13452.630153
Audi,275,11500.00,18240.624364
Mercedes-Benz,517,14500.00,31170.234043
BMW,428,15200.00,23868.789136
Toyota,344,16736.25,22346.836628


,count,median,mean
Body,,,
hatch,173,8300.0,10374.849249
vagon,240,8600.0,10250.451542
van,393,9100.0,11443.570941
sedan,1005,9999.0,15995.819234
other,277,12800.0,23481.151625
crossover,550,25500.0,36574.078473


,count,median,mean
Engine Type,,,
Gas,395,10500.0,12503.762658
Other,94,10575.0,18286.925532
Diesel,1182,11200.0,20142.203190
Petrol,967,12900.0,21698.096939


,count,median,mean
Registration,,,
no,238,3100.0,4174.348739
yes,2400,12700.0,21022.752567


## Redundancy and action plan
The original uncentered VIF calculation produced a high Year score and led to dropping it. I inspect centered VIF with an intercept and retain model-year information as age rather than removing it solely from that old calculation. Endogeneity, independence and constant variance require evidence; they are not guaranteed by this cross-sectional dataset.


In [8]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
variables = eda[["Year", "Mileage", "EngineV"]].copy()
variables.loc[(variables.EngineV <= 0) | (variables.EngineV > 6.5), "EngineV"] = np.nan
variables = variables.dropna()
design = sm.add_constant(variables - variables.mean())
display(pd.DataFrame({"feature": variables.columns, "vif": [variance_inflation_factor(design.to_numpy(), i + 1) for i in range(len(variables.columns))]}))
px.imshow(variables.corr(), zmin=-1, zmax=1, color_continuous_scale="RdBu_r", title="Training predictor correlations").show()


,feature,vif
0,Year,1.819732
1,Mileage,1.818991
2,EngineV,1.001316


1. Exclude rows without a valid positive target and deduplicate exact source records before splitting.
2. Preserve row IDs for prediction-to-actual comparison.
3. Treat suspicious measurements as missing, then learn imputation and encoding inside CV.
4. Compare the original linear, XGBoost, forest and decision-tree families with a median baseline and regularized/boosting alternatives.
5. Inspect overfitting, price-unit errors and percentage errors before making a decision.

**My next observations:** explain which hypotheses merit controlled experiments.
